# OASIS-2 V3 Temporal Reliability Colab

This notebook trains and evaluates the `oasis2_multimodal_v3_temporal` candidate. It targets temporal paradox reduction while keeping OASIS-1 as the active fallback and OASIS-2 candidate-only until strict promotion gates pass.

**No ONNX export is run here.** Export stays blocked until promotion gates pass.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_multimodal_v3_temporal'
TRAIN_CONFIG = 'configs/oasis2_train_multimodal_v3_temporal.yaml'
REPO_URL = 'https://github.com/Billrichard209/Cerebrasense-.git'
MIN_COMMIT = '77eea77519a3743631917424e0a232cc1fa5b74f'

ACCEPTANCE = {
    'auroc_min_exclusive': 0.725,
    'balanced_accuracy_min': 0.68,
    'subject_auroc_min': 0.72,
    'review_required_max_exclusive': 31,
    'temporal_paradox_max': 1,
}

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(f"{name}: {'[OK]' if path.exists() else '[MISSING]'} {path}")

if not OASIS2_BUNDLE_ROOT.exists():
    raise FileNotFoundError(f'Missing OASIS-2 bundle: {OASIS2_BUNDLE_ROOT}')

## Clone Latest Repo and Install Colab Requirements

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
PYTHON = sys.executable

LOG_ROOT = RUNTIME_ROOT / 'logs' / RUN_NAME
LOG_ROOT.mkdir(parents=True, exist_ok=True)

def run_live(cmd, *, cwd=None, label=None, allow_fail=False):
    step = label or Path(str(cmd[0])).stem
    printable = ' '.join(str(part) for part in cmd)
    log_path = LOG_ROOT / f'{step}.log'
    print(f"\nRUNNING {step}: {printable}", flush=True)
    print(f"LOG_FILE={log_path}", flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f"\n\n=== {step} ===\n{printable}\n")
        process = subprocess.Popen(
            cmd,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    print(f"RETURN_CODE {step}: {return_code}", flush=True)
    if return_code != 0 and not allow_fail:
        raise RuntimeError(f"Command failed ({step}): {printable}")
    return return_code

for stale_root in [Path('/content/cerebrasense'), Path('/content/Cerebrasense-')]:
    if stale_root.exists():
        shutil.rmtree(stale_root)

run_live(['git', 'clone', REPO_URL, str(REPO_ROOT)], cwd='/content', label='git-clone')
run_live(['git', 'checkout', 'main'], cwd=REPO_ROOT, label='git-checkout-main')
repo_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
ancestor = subprocess.run(['git', 'merge-base', '--is-ancestor', MIN_COMMIT, 'HEAD'], cwd=REPO_ROOT)
if ancestor.returncode != 0:
    raise RuntimeError(f'Cloned commit {repo_commit} is older than required {MIN_COMMIT}')
print(f'Active commit: {repo_commit}')

run_live([PYTHON, '-u', '-m', 'pip', 'install', '-r', str(BACKEND_ROOT / 'requirements-colab.txt')], cwd=REPO_ROOT, label='pip-install')

## Preflight: GPU, Runtime Roots, and Config

In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU not detected. In Colab, choose Runtime > Change runtime type > T4 GPU.')

os.chdir(BACKEND_ROOT)
os.environ['ALZ_DATA_ROOT'] = str(RUNTIME_ROOT / 'data')
os.environ['ALZ_OUTPUTS_ROOT'] = str(RUNTIME_ROOT / 'outputs')
os.environ['ALZ_WORKSPACE_ROOT'] = str(DRIVE_ROOT)
os.environ['PYTHONUNBUFFERED'] = '1'

train_config_path = BACKEND_ROOT / TRAIN_CONFIG
if not train_config_path.exists():
    raise FileNotFoundError(f'Missing train config: {train_config_path}')

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"ALZ_DATA_ROOT={os.environ['ALZ_DATA_ROOT']}")
print(f"ALZ_OUTPUTS_ROOT={os.environ['ALZ_OUTPUTS_ROOT']}")
print(f"TRAIN_CONFIG={train_config_path}")
run_live(['nvidia-smi'], cwd=BACKEND_ROOT, label='nvidia-smi', allow_fail=True)
run_live([PYTHON, '-u', 'scripts/train_oasis2_colab.py', '--help'], cwd=BACKEND_ROOT, label='train-script-import-smoke')

## Train V3 Temporal Candidate

In [ ]:
os.chdir(BACKEND_ROOT)

train_cmd = [
    PYTHON, '-u', 'scripts/train_oasis2_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--bundle-root', str(OASIS2_BUNDLE_ROOT),
    '--run-name', RUN_NAME,
    '--config', TRAIN_CONFIG,
    '--device', 'auto',
]
run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
print(f"Live train log: {LOG_ROOT / 'train-v3-temporal.log'}")
print(f"Epoch metrics CSV: {run_root / 'metrics' / 'epoch_metrics.csv'}")
run_live(train_cmd, cwd=BACKEND_ROOT, label='train-v3-temporal')

checkpoint_path = run_root / 'checkpoints' / 'best_model.pt'
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Missing checkpoint after training: {checkpoint_path}')
print(f'BEST_CHECKPOINT={checkpoint_path}')

## Evaluate and Calibrate Candidate

In [ ]:
os.chdir(BACKEND_ROOT)
run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
checkpoint_path = run_root / 'checkpoints' / 'best_model.pt'
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')

eval_cmd = [
    PYTHON, '-u', 'scripts/evaluate_oasis2_candidate.py',
    '--run-name', RUN_NAME,
    '--checkpoint-path', str(checkpoint_path),
    '--model-config-path', 'configs/oasis2_multimodal_model.yaml',
    '--device', 'cuda',
    '--selection-metric', 'balanced_accuracy',
    '--batch-size', '1',
    '--num-workers', '0',
    '--cache-rate', '0.0',
    '--image-size', '96', '96', '96',
]
run_live(eval_cmd + ['--max-batches', '2'], cwd=BACKEND_ROOT, label='smoke-eval-2-batches')
run_live(eval_cmd, cwd=BACKEND_ROOT, label='full-evaluate-calibrate-v3')

## Refresh Evidence and Export Evaluated Run

In [ ]:
import shutil

os.chdir(BACKEND_ROOT)
run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
predictions_csv = run_root / 'evaluation' / 'post_train_test_best_model' / 'predictions.csv'
audit_json = RUNTIME_ROOT / 'outputs' / 'reports' / 'longitudinal' / f'audit_{RUN_NAME}.json'
audit_json.parent.mkdir(parents=True, exist_ok=True)

if predictions_csv.exists():
    run_live([
        PYTHON, '-u', 'scripts/audit_temporal_paradoxes.py',
        '--predictions-csv', str(predictions_csv),
        '--output-json', str(audit_json),
    ], cwd=BACKEND_ROOT, label='temporal-paradox-audit')
else:
    raise FileNotFoundError(f'Missing predictions for temporal audit: {predictions_csv}')

run_live([PYTHON, '-u', 'scripts/analyze_oasis2_mixed_label_errors.py', '--run-name', RUN_NAME], cwd=BACKEND_ROOT, label='mixed-label-analysis')
run_live([PYTHON, '-u', 'scripts/build_oasis2_leaderboard.py', '--workspace-root', str(DRIVE_ROOT)], cwd=BACKEND_ROOT, label='leaderboard')

drive_frontend_payload = DRIVE_ROOT / 'frontend_demo' / 'data' / 'research_mode.json'
drive_frontend_payload.parent.mkdir(parents=True, exist_ok=True)
run_live([
    PYTHON, '-u', 'scripts/build_cerebrasense_control_tower.py',
    '--frontend-payload-path', str(drive_frontend_payload),
], cwd=BACKEND_ROOT, label='control-tower-refresh')

zip_base = DRIVE_ROOT / f'{RUN_NAME}_evaluated'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', run_root))
print(f'EVALUATED_RUN_ZIP={zip_path}')
print(f'DRIVE_RESEARCH_MODE_JSON={drive_frontend_payload}')

## Final Metric Summary

In [ ]:
import json
import pandas as pd

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
summary_path = run_root / 'reports' / 'colab_run_summary.json'
metrics_csv = run_root / 'metrics' / 'epoch_metrics.csv'
calibrated_metrics_path = run_root / 'evaluation' / 'post_train_test_best_model_threshold_balanced_accuracy' / 'metrics.json'
raw_metrics_path = run_root / 'evaluation' / 'post_train_test_best_model' / 'metrics.json'
payload_path = DRIVE_ROOT / 'frontend_demo' / 'data' / 'research_mode.json'
audit_json = RUNTIME_ROOT / 'outputs' / 'reports' / 'longitudinal' / f'audit_{RUN_NAME}.json'

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(f"Training Summary for {RUN_NAME}:")
    print(f"- Best Epoch: {summary.get('best_epoch')}")
    best_value = summary.get('best_monitor_value')
    print(f"- Best Val AUROC: {best_value:.4f}" if isinstance(best_value, (float, int)) else f"- Best Val AUROC: {best_value}")
    print(f"- Best Checkpoint: {summary.get('best_checkpoint')}")

metrics = json.loads((calibrated_metrics_path if calibrated_metrics_path.exists() else raw_metrics_path).read_text(encoding='utf-8'))
payload = json.loads(payload_path.read_text(encoding='utf-8')) if payload_path.exists() else {}
audit = json.loads(audit_json.read_text(encoding='utf-8')) if audit_json.exists() else {}
candidate = payload.get('oasis2_candidate', {})
paradox_summary = payload.get('paradox_summary', {})

observed = {
    'auroc': metrics.get('auroc'),
    'balanced_accuracy': metrics.get('balanced_accuracy'),
    'specificity': metrics.get('specificity'),
    'f1': metrics.get('f1'),
    'review_required_count': candidate.get('review_required_count', metrics.get('review_required_count')),
    'subject_consensus_auroc': candidate.get('subject_consensus_auroc'),
    'temporal_paradox_count': paradox_summary.get('temporal_paradox_count', audit.get('paradox_count')),
}

print('\nFinal calibrated/evidence metrics:')
for key, value in observed.items():
    print(f'- {key}: {value}')

checks = {
    'auroc_beats_v1': observed['auroc'] is not None and observed['auroc'] > ACCEPTANCE['auroc_min_exclusive'],
    'balanced_accuracy_ok': observed['balanced_accuracy'] is not None and observed['balanced_accuracy'] >= ACCEPTANCE['balanced_accuracy_min'],
    'subject_auroc_ok': observed['subject_consensus_auroc'] is not None and observed['subject_consensus_auroc'] >= ACCEPTANCE['subject_auroc_min'],
    'review_burden_ok': observed['review_required_count'] is not None and observed['review_required_count'] < ACCEPTANCE['review_required_max_exclusive'],
    'temporal_paradoxes_ok': observed['temporal_paradox_count'] is not None and observed['temporal_paradox_count'] <= ACCEPTANCE['temporal_paradox_max'],
}
print('\nAcceptance checks:')
for key, passed in checks.items():
    print(f"- {key}: {'PASS' if passed else 'BLOCKED'}")

if metrics_csv.exists():
    print('\nLast 5 epochs:')
    print(pd.read_csv(metrics_csv).tail())